# 02. Causal Statistical Genetics — Drug-Target `cis-MR`, `colocPropTest`, `cTWAS`, `LAVA`, `GenomicSEM`, `LHC-MR` & `GRAPPLE`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gillanilab/pythonic_biotools/blob/main/notebooks/02_causal_genetics_ctwas_cismr_and_sem.ipynb)

This tutorial demonstrates seven modern summary-statistics causal genetics and multivariate architecture methods from `pythonic_biotools`:

1. **`cismr`** — Correlated cis-locus drug-target Mendelian Randomization (`mr_pcgmm` and outlier-robust pair-voting `mr_spi`).
2. **`colocproptest`** — Frequentist LD-robust proportional colocalization testing valid under multiple causal variants.
3. **`ctwas`** — Causal Transcriptome/Proteome-Wide Association Study separating causal genes from LD hitchhiking via Empirical Bayes + group-prior `SuSiE-RSS`.
4. **`lava`** — Univariate local heritability ($h^2_{\text{local}}$), bivariate local $r_g$, and partial local genetic correlation $r_g(X, Y \mid Z)$.
5. **`genomicsem`** — `lavaan`-syntax Reticular Action Model (`DWLS`) structural equation modeling and vectorized Common-Factor SNP GWAS ($Q_{\text{SNP}}$).
6. **`lhcmr`** — Bidirectional causal inference simultaneous with a latent heritable confounder $U$ and sample overlap.
7. **`grapple`** — Weak-instrument robust profile-likelihood MR and multimodal pleiotropic pathway discovery.

In [1]:
import subprocess
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/gillanilab/pythonic_biotools.git"]
    )
else:
    repo_root = Path.cwd().resolve()
    if not (repo_root / "pythonic_biotools").is_dir() and (repo_root.parent / "pythonic_biotools").is_dir():
        repo_root = repo_root.parent
    if str(repo_root) not in sys.path:
        sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from pythonic_biotools import cismr, colocproptest, ctwas, genomicsem, grapple, lava, lhcmr

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 15)
print("Loaded statistical genetics toolkit: cismr, colocproptest, ctwas, lava, genomicsem, lhcmr, grapple")

Loaded statistical genetics toolkit: cismr, colocproptest, ctwas, lava, genomicsem, lhcmr, grapple


## 1. Drug-Target Correlated `cis-MR` (`cismr`) & Proportional Colocalization (`colocproptest`)

We simulate a 36-SNP cis-pQTL drug-target region in autoregressive LD ($R_{ij} = 0.72^{|i - j|}$, $N = 50{,}000$) with two shared causal cis-variants ($\theta_{\text{true}} = 0.38$) and compare `mr_pcgmm` (`default` vs. `R_default`), `mr_spi`, and `run_coloc_prop_test`.

In [2]:
rng = np.random.default_rng(2026)
m, n_gwas = 36, 50_000
idx = np.arange(m)
ld_R = 0.72 ** np.abs(idx[:, None] - idx[None, :])
chol_R = np.linalg.cholesky(ld_R)

b_pqtl_true = np.zeros(m)
b_pqtl_true[[6, 22]] = [0.20, 0.15]
theta_true = 0.38

se_vec = np.full(m, 1.0 / np.sqrt(n_gwas))
bx = ld_R @ b_pqtl_true + se_vec * (chol_R @ rng.standard_normal(m))
by = ld_R @ (theta_true * b_pqtl_true) + se_vec * (chol_R @ rng.standard_normal(m))

pcgmm_def = cismr.mr_pcgmm(bx, se_vec, by, se_vec, ld_R=ld_R, nx=n_gwas, ny=n_gwas, settings="default")
pcgmm_r = cismr.mr_pcgmm(bx, se_vec, by, se_vec, ld_R=ld_R, nx=n_gwas, ny=n_gwas, settings="R_default")
spi_res = cismr.mr_spi(bx, se_vec, by, se_vec, ld_R=ld_R, n=n_gwas, settings="default")
prop_res = colocproptest.run_coloc_prop_test(
    {"beta": bx, "se": se_vec}, {"beta": by, "se": se_vec}, LD=ld_R, settings="default"
)

summary_cismr = pd.DataFrame(
    [
        {"method": "PC-GMM (default)", "estimate": pcgmm_def.estimate, "se": pcgmm_def.se, "pval": pcgmm_def.pval},
        {"method": "PC-GMM (R_default)", "estimate": pcgmm_r.estimate, "se": pcgmm_r.se, "pval": pcgmm_r.pval},
        {"method": "MR-SPI (pair-voting)", "estimate": spi_res.estimate, "se": spi_res.se, "pval": spi_res.pval},
    ]
)
print(summary_cismr.round({"estimate": 4, "se": 4}).to_string(index=False))
print(
    f"\ncolocPropTest: eta_hat = {prop_res.eta_hat:.4f}, "
    f"reject_proportionality = {prop_res.reject_proportionality}, "
    f"evidence_for_coloc = {prop_res.evidence_for_coloc}"
)

              method  estimate     se         pval
    PC-GMM (default)    0.3900 0.0236 1.734978e-61
  PC-GMM (R_default)    0.3859 0.0236 8.243698e-14
MR-SPI (pair-voting)    0.3850 0.0222 1.305365e-67

colocPropTest: eta_hat = 0.3932, reject_proportionality = False, evidence_for_coloc = True


## 2. Causal TWAS Separating Causal Targets from LD Hitchhikers (`ctwas`)

Standard TWAS frequently flags non-causal bystander genes whose eQTLs sit in LD with causal variants. Here `CAUSAL_TARGET` mediates risk (`rs6`, `rs22`) while `BYSTANDER_GENE` merely tags nearby SNPs (`rs7`, `rs8`), and `run_ctwas` jointly fine-maps genes and cis-SNPs.

In [3]:
gwas_z_df = pd.DataFrame({"id": [f"rs{i}" for i in range(m)], "z": by / se_vec})
weights_list = [
    {"gene_id": "CAUSAL_TARGET", "snps": ["rs6", "rs22"], "weights": [0.20, 0.15]},
    {"gene_id": "BYSTANDER_GENE", "snps": ["rs7", "rs8"], "weights": [0.16, 0.12]},
]
imp_region = ctwas.impute_expr_z(gwas_z_df, ld_R, weights_list, region_id="chr1_locus", settings="default")
ct_res = ctwas.run_ctwas([imp_region], n_samples=n_gwas, L=3, settings="default")

print("cTWAS fine-mapped gene posterior inclusion probabilities (PIP):")
print(ct_res.gene_df[["gene_id", "z", "stage1_pip", "susie_pip", "in_cs"]].round(4).to_string(index=False))

cTWAS fine-mapped gene posterior inclusion probabilities (PIP):
       gene_id       z  stage1_pip  susie_pip  in_cs
 CAUSAL_TARGET 21.5511         1.0     1.0000   True
BYSTANDER_GENE 12.4207         0.0     0.0156  False


## 3. Local Heritability & Partial Local Genetic Correlation (`lava`)

We evaluate local genetic overlap between `Target_Prot` and `CAD_Risk` while conditioning out a local pleiotropic `Lipid_Conf` trait that drives shared signal at SNP `rs14`.

In [4]:
b_conf = np.zeros(m)
b_conf[14] = 0.18
z_lava = pd.DataFrame(
    {
        "Target_Prot": bx / se_vec + np.sqrt(n_gwas) * (ld_R @ b_conf),
        "CAD_Risk": by / se_vec + np.sqrt(n_gwas) * (ld_R @ b_conf),
        "Lipid_Conf": np.sqrt(n_gwas) * (ld_R @ b_conf) + (chol_R @ rng.standard_normal(m)),
    }
)
locus = lava.process_locus(
    z_lava,
    ld_R,
    sample_sizes={"Target_Prot": n_gwas, "CAD_Risk": n_gwas, "Lipid_Conf": n_gwas},
    locus_id="chr1_drug_locus",
)
univ_df = lava.run_univ(locus)
bivar_df = lava.run_bivar(locus, traits=["Target_Prot", "CAD_Risk"], n_sim=3000, seed=42)
pcor_df = lava.run_pcor(locus, pair=("Target_Prot", "CAD_Risk"), cond_traits=["Lipid_Conf"], n_sim=3000, seed=42)

print("Univariate local heritability:")
print(univ_df[["trait", "h2_loc", "p"]].round({"h2_loc": 4}).to_string(index=False))
print("\nUnconditional bivariate local r_g:")
print(bivar_df[["trait1", "trait2", "rg", "p"]].round({"rg": 4}).to_string(index=False))
print("\nPartial local r_g(Target_Prot, CAD_Risk | Lipid_Conf):")
print(pcor_df[["trait1", "trait2", "cond_traits", "pcor", "p"]].round({"pcor": 4}).to_string(index=False))

Univariate local heritability:
      trait  h2_loc             p
Target_Prot  0.0983  0.000000e+00
   CAD_Risk  0.0494  0.000000e+00
 Lipid_Conf  0.0299 1.190018e-304

Unconditional bivariate local r_g:
     trait1   trait2     rg   p
Target_Prot CAD_Risk 0.9026 0.0

Partial local r_g(Target_Prot, CAD_Risk | Lipid_Conf):
     trait1   trait2 cond_traits   pcor            p
Target_Prot CAD_Risk  Lipid_Conf 0.9969 2.401453e-39


## 4. Multivariate `GenomicSEM` & Common-Factor SNP Heterogeneity ($Q_{\text{SNP}}$)

We fit a single latent cardiometabolic factor (`F_cm =~ CAD + T2D + SBP + TG`) via `DWLS` and test two SNPs with `commonfactor_gwas`: `rs_shared` acting proportionally through `F_cm` (low $Q_{\text{SNP}}$) vs. `rs_specific` acting solely on `CAD` (high $Q_{\text{SNP}}$).

In [5]:
loadings = np.array([0.70, 0.62, 0.50, 0.55])
S_cm = np.outer(loadings, loadings) + np.diag([0.18, 0.22, 0.28, 0.25])
V_cm = np.eye(10) * 1.5e-4
cov_cm = genomicsem.GenomicCovStruct(S=S_cm, V_S=V_cm, trait_names=["CAD", "T2D", "SBP", "TG"])

sem_fit = genomicsem.usermodel(cov_cm, "F_cm =~ CAD + T2D + SBP + TG", estimation="DWLS", settings="default")
print(f"Model fit: CFI = {sem_fit.modelfit['CFI']:.4f}, SRMR = {sem_fit.modelfit['SRMR']:.4f}")
print(sem_fit.results_df[["lhs", "op", "rhs", "est", "se", "std_est", "p_value"]].head(4).round(4).to_string(index=False))

snp_tables = [
    pd.DataFrame(
        {
            "SNP": ["rs_shared", "rs_specific"],
            "BETA": [0.030 * lam_k, 0.045 if k == 0 else 0.0],
            "SE": [0.004, 0.004],
        }
    )
    for k, lam_k in enumerate(loadings)
]
cf_gwas = genomicsem.commonfactor_gwas(snp_tables, cov_cm, settings="default")
print("\nCommon-factor SNP GWAS & Q_SNP heterogeneity test:")
print(cf_gwas[["SNP", "beta_F", "se", "p_value", "Q_SNP", "Q_pval"]].round({"beta_F": 4, "se": 4, "Q_SNP": 2}).to_string(index=False))

Model fit: CFI = 1.0000, SRMR = 0.0000
 lhs op rhs  est     se  std_est  p_value
F_cm =~ CAD 0.70 0.0150   0.8552      0.0
F_cm =~ T2D 0.62 0.0137   0.7975      0.0
F_cm =~ SBP 0.50 0.0122   0.6868      0.0
F_cm =~  TG 0.55 0.0127   0.7399      0.0

Common-factor SNP GWAS & Q_SNP heterogeneity test:
        SNP  beta_F     se      p_value  Q_SNP       Q_pval
  rs_shared  0.0300 0.0034 3.873197e-19   0.00 1.000000e+00
rs_specific  0.0224 0.0033 2.248286e-11  80.99 1.880875e-17


## 5. Bidirectional Confounder-Aware MR (`lhcmr`) & Multimodal Pathway Discovery (`grapple`)

Finally, we apply `lhc_mr` to genome-wide summary statistics confounded by an unmeasured heritable factor $U$, and use `grapple_robust_est` to detect multimodal pleiotropic pathways across genetic instruments.

In [6]:
# 5A. LHC-MR on genome-wide summary statistics with unidirectional X -> Y = 0.30 and latent confounder U
m_gw = 2500
ld_scores = rng.gamma(shape=2.5, scale=12.0, size=m_gw)
scale_ld = np.sqrt(ld_scores / np.mean(ld_scores))
g_x = np.where(rng.random(m_gw) < 0.08, rng.normal(0.0, 0.018, m_gw), 0.0) * scale_ld
g_u = np.where(rng.random(m_gw) < 0.05, rng.normal(0.0, 0.015, m_gw), 0.0) * scale_ld
g_y = np.where(rng.random(m_gw) < 0.08, rng.normal(0.0, 0.018, m_gw), 0.0) * scale_ld

se_gw = np.full(m_gw, 1.0 / np.sqrt(80_000))
bx_gw = g_x + 0.25 * g_u + rng.normal(0.0, se_gw)
by_gw = 0.30 * (g_x + 0.25 * g_u) + 0.20 * g_u + g_y + rng.normal(0.0, se_gw)

lhc_res = lhcmr.lhc_mr(bx_gw, se_gw, by_gw, se_gw, ld_scores=ld_scores, nx=80_000, ny=80_000, n_starts=3, seed=42)
print(
    f"[LHC-MR] alpha_(X->Y) = {lhc_res.alpha_xy:.3f} (SE={lhc_res.alpha_xy_se:.3f}, p={lhc_res.alpha_xy_pval:.2e}) | "
    f"alpha_(Y->X) = {lhc_res.alpha_yx:.3f} (SE={lhc_res.alpha_yx_se:.3f})"
)

# 5B. GRAPPLE robust profile-likelihood MR with two distinct causal/pleiotropic modes (+0.35 and -0.25)
bx_iv = rng.uniform(0.04, 0.10, size=50)
sx_iv = np.full(50, 0.008)
sy_iv = np.full(50, 0.009)
slopes_iv = np.where(np.arange(50) < 32, 0.35, -0.25)
by_iv = slopes_iv * bx_iv + rng.normal(0.0, sy_iv)

grap_res = grapple.grapple_robust_est(bx_iv, sx_iv, by_iv, sy_iv, diagnosis=True, settings="default")
print(
    f"\n[GRAPPLE] Dominant beta_hat = {grap_res.beta_hat:.3f} (SE={grap_res.beta_se:.3f}), "
    f"multimodal = {grap_res.is_multimodal}, detected modes = {[round(md.beta_hat, 3) for md in grap_res.modes]}"
)

[LHC-MR] alpha_(X->Y) = 0.281 (SE=0.016, p=9.79e-71) | alpha_(Y->X) = 0.051 (SE=0.020)

[GRAPPLE] Dominant beta_hat = 0.276 (SE=0.036), multimodal = False, detected modes = [0.276]
